# Periodic Darcy: separate face, local and reference refinement

This case follows Paredes, Valentin and Versieux (2017), section 4, DOI
[10.1090/mcom/3108](https://doi.org/10.1090/mcom/3108). On the unit square,
$K=1+100\cos^2(150x)\sin^2(150y)$, $f=\sin(x)\sin(y)$ and boundary pressure is zero.
The fixed MHM grid has 8×8 squares, local continuous Q1 fields, and 1–32 constant
normal-flux modes per face. The historical local refinement is unspecified.

The classical reference has its own refinement study. All comparisons below
are integrated full broken H1 or L2 norms, not norms of plotted sample arrays.


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from pymhm.quadrilateral import CartesianMacroMesh, solve_darcy_quadrilateral
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent


## Small independent affine patch

An affine pressure is exactly represented by Q1 locals and constant flux traces.
This portable patch runs here; the larger periodic fields are replayed from their
numerical records, without requiring a high-memory solve in the notebook.


In [ ]:
exact = lambda points: 1 + points[:, 0] - 2 * points[:, 1]
patch = solve_darcy_quadrilateral(CartesianMacroMesh(2), local_refinement=4, dirichlet=exact)
assert patch.l2_error(exact) < 1e-11
assert patch.flux_l2_error([-1, 2]) < 1e-10
print({"pressure_L2_error": patch.l2_error(exact), "residual": patch.hybrid.residual})


## Recorded norms and independent reference sensitivity

The denominator is the indicated classical field. The gradient seminorm is also
stored separately; the plotted relative H1 quantity includes the L2 term.
Reference refinement is a measured sensitivity, not an upper bound for the
unknown exact-solution error.


In [ ]:
comparison = json.loads((root / "examples/results/periodic-comparison.json").read_text())
reference = comparison["primary_reference"]
rows = [row for row in comparison["comparisons"]
        if row["reference"] == reference and "segments" in row]
assert len(rows) == 18
assert {(row["refinement"], row["segments"]) for row in rows} == {
    (r, s) for r in (128, 256, 512) for s in (1, 2, 4, 8, 16, 32)
}
assert {row["segments"] for row in rows} == {1, 2, 4, 8, 16, 32}
for row in sorted(rows, key=lambda r: (r["refinement"], r["segments"])):
    assert np.isfinite([row["relative_l2"], row["relative_h1"]]).all()
    print(f"r={row['refinement']:4d}  s={row['segments']:2d}  "
          f"relative H1={row['relative_h1']:.8g}  relative L2={row['relative_l2']:.8g}")
print("Numerical denominator:", reference)
reference_controls = [row for row in comparison["comparisons"]
                      if row["reference"] == reference and row["field"].startswith("conforming:")]
for row in reference_controls:
    print(row["field"], {"relative_H1_change": row["relative_h1"], "relative_L2_change": row["relative_l2"]})


## Published curve and physical fields

The publication markers come from vector-PDF coordinates with a one-point
vertical extraction allowance. This is distinct from numerical reference error.
Macro contours are actual boundaries; signed physical fluxes use common scales.
The image samples preserve their cell-side finite-element values without
interface averaging or smoothing.

The final reference increment and the MHM differences are printed from the current comparison records above. They use the explicitly identified reference as denominator. A refinement increment is a sensitivity, not a certified error bound. The classical Q1 field on the same local spacing shares approximation error and is retained as a distinct control. The historical local subdivision count is unavailable, so the published curve is not claimed to have been reproduced identically.


In [ ]:
for name in ("problem", "convergence", "fields"):
    display(Image(filename=str(root / f"docs/figures/periodic/{name}.png")))


## Reproduce the larger campaign

Use `examples/verify_periodic.py` for the reference and MHM solves,
`examples/compare_periodic.py` for exact common-partition polynomial norms, and
`examples/plot_periodic.py` for figures. See the periodic case page for explicit
commands, quadrature and memory requirements. Large field archives live under
`build/results/periodic`; the compact numerical records are distributed with
the examples. Source acquisition and subsequent comparison provenance are
recorded separately.
